# Binary classification

In [1]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torchvision.transforms.v2 as v2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import os

from sklearn.model_selection import train_test_split

import time

from PIL import Image

In [2]:
# Importing the datasets
dataset_dir = "Datasets"

images = []
labels = []

for label_name in os.listdir(dataset_dir):

    # On transforme le nom du dossier en label numérique
    if label_name == "Photo_resized":
        label = 1
    else:
        label = 0

    label_dir = os.path.join(dataset_dir, label_name + "_resized")
    if os.path.isdir(label_dir):
        for image_name in os.listdir(label_dir):
            image_path = os.path.join(label_dir, image_name)
            images.append(image_path)
            labels.append(label)


In [3]:
# Splitting the dataset into training and testing sets
train_images, test_images, train_labels, test_labels = train_test_split(images, labels, test_size=0.2, random_state=42, stratify=labels)

print(f"Number of training images: {len(train_images)}")
print(f"Number of testing images: {len(test_images)}")

print(f"Number of training labels: {len(train_labels)}")
print(f"Number of testing labels: {len(test_labels)}")

Number of training images: 33118
Number of testing images: 8280
Number of training labels: 33118
Number of testing labels: 8280


In [17]:
# Creating the model

class BinaryClassifier(nn.Module):
    def __init__(self):
        super(BinaryClassifier, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 32 * 32, 128)
        self.fc2 = nn.Linear(128, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = self.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

In [18]:
# Creating the dataset class
class ImageDataset(Dataset):

    def __init__(self, image_paths, labels):
        self.image_paths = image_paths
        self.labels = labels

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image = Image.open(
            self.image_paths[idx]
        ).convert("L")  # <-- 1-channel grayscale

        image = torch.from_numpy(
            np.array(image)
        ).unsqueeze(0).float() / 255.0

        label = torch.tensor(
            self.labels[idx],
            dtype=torch.float32
        )

        return image, label

In [19]:
# Creating the training and testing datasets
train_dataset = ImageDataset(
    train_images,
    train_labels
)

test_dataset = ImageDataset(
    test_images,
    test_labels
)

In [20]:
# Creating the data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [21]:
images_batch, labels_batch = next(iter(train_loader))

print(images_batch.shape)
print(labels_batch.shape)

torch.Size([32, 1, 128, 128])
torch.Size([32])


In [22]:
# training the model

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

model = BinaryClassifier().to(device)

criterion = nn.BCELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

num_epochs = 1

for epoch in range(num_epochs):

    start_time = time.time()

    print(f"--- Epoch {epoch+1}/{num_epochs} ---")

    # -------------------------
    # Training
    # -------------------------
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    nb_batches = len(train_loader)
    current_batch = 0

    for images_batch, labels_batch in train_loader:

        if current_batch >= nb_batches:
            current_batch = 0
        else:
            current_batch += 1

        print("Batch " + str(current_batch) + "/" + str(nb_batches))
        
        

        images_batch = images_batch.to(device)
        labels_batch = labels_batch.to(device)

        # Forward
        outputs = model(images_batch).squeeze(1)

        # Loss
        loss = criterion(outputs, labels_batch)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images_batch.size(0)

        predictions = (outputs >= 0.5).float()

        train_correct += (
            predictions == labels_batch
        ).sum().item()

        train_total += labels_batch.size(0)

    end_time = time.time()

    print(f"Epoch {epoch+1} completed in {end_time - start_time:.2f} seconds.")

    train_loss /= train_total
    train_accuracy = train_correct / train_total

    # -------------------------
    # Testing
    # -------------------------
    model.eval()

    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():

        for images_batch, labels_batch in test_loader:

            images_batch = images_batch.to(device)
            labels_batch = labels_batch.to(device)

            outputs = model(images_batch).squeeze(1)

            loss = criterion(outputs, labels_batch)

            test_loss += loss.item() * images_batch.size(0)

            predictions = (outputs >= 0.5).float()

            test_correct += (
                predictions == labels_batch
            ).sum().item()

            test_total += labels_batch.size(0)

    test_loss /= test_total
    test_accuracy = test_correct / test_total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_accuracy:.4f} "
        f"Test Loss: {test_loss:.4f} "
        f"Test Acc: {test_accuracy:.4f}"
    )

--- Epoch 1/1 ---
Batch 1/1035
Batch 2/1035
Batch 3/1035
Batch 4/1035
Batch 5/1035
Batch 6/1035
Batch 7/1035
Batch 8/1035
Batch 9/1035
Batch 10/1035
Batch 11/1035
Batch 12/1035
Batch 13/1035
Batch 14/1035
Batch 15/1035
Batch 16/1035
Batch 17/1035
Batch 18/1035
Batch 19/1035
Batch 20/1035
Batch 21/1035
Batch 22/1035
Batch 23/1035
Batch 24/1035
Batch 25/1035
Batch 26/1035
Batch 27/1035
Batch 28/1035
Batch 29/1035
Batch 30/1035
Batch 31/1035
Batch 32/1035
Batch 33/1035
Batch 34/1035
Batch 35/1035
Batch 36/1035
Batch 37/1035
Batch 38/1035
Batch 39/1035
Batch 40/1035
Batch 41/1035
Batch 42/1035
Batch 43/1035
Batch 44/1035
Batch 45/1035
Batch 46/1035
Batch 47/1035
Batch 48/1035
Batch 49/1035
Batch 50/1035
Batch 51/1035
Batch 52/1035
Batch 53/1035
Batch 54/1035
Batch 55/1035
Batch 56/1035
Batch 57/1035
Batch 58/1035
Batch 59/1035
Batch 60/1035
Batch 61/1035
Batch 62/1035
Batch 63/1035
Batch 64/1035
Batch 65/1035
Batch 66/1035
Batch 67/1035
Batch 68/1035
Batch 69/1035
Batch 70/1035
Batch 71/10